# Countdown judge from the SFT start: corrupted reasoning, DPO judge, self-judge check and calibration

Part 1 of 2 (part 2: Consistency_GRPO_Judge_Countdown, same [config]). S = Consistency_SFT_Countdown's adapter (Qwen3-0.6B by
default, MODEL_KEY). Label-free: the checker only measures.
1. **[build]** The GRPO trainer on S with the two-pass rollout (Consistency_GRPO_Olmo_Judge's): its vLLM engine samples
   everything here, and the calibration uses the training rollout itself. Nothing is trained by GRPO in this notebook.
2. **[corruption]** S's normal completion per corruption puzzle (1,000 four- + 1,000 three-number), then S's solution after
   corrupted reasoning: empty, and the first 10 / 25 / 50% of its own reasoning.
3. **[pairs] / [judge DPO]** chosen = the normal solution + answer, rejected = the one after corrupted reasoning; DPO
   continues S (reference S) on TRAIN_KINDS (the OLMo judge's recipe), saved to JUDGE_PATH.
4. **[calibrate]** S's two-pass rollouts on the grade puzzles scored by the judge, S itself and the untrained model:
   within-puzzle win rate of correct over wrong (the self-judge check); the judge's reward weight and missing value,
   saved to CALIB_PATH for part 2.

Each stage is skipped when its output exists. Run order: Run all ([install] needs no restart).

In [ ]:
# [config] Everything you choose lives here.
MODELS = {   # key: HF id, chat-template kwargs (Qwen3: thinking off, as the SFT and the screens)
    "qwen3-0.6b": ("Qwen/Qwen3-0.6B", {"enable_thinking": False}),
    "qwen3-1.7b": ("Qwen/Qwen3-1.7B", {"enable_thinking": False}),
}
MODEL_KEY = "qwen3-0.6b"
MODEL, CHAT_KWARGS = MODELS[MODEL_KEY]
SEED = 0
SMOKE = False                    # True: every cell end to end on a few puzzles, 2 DPO steps, 2 GRPO steps per arm (-smoke)
# S: Consistency_SFT_Countdown's run (same settings there)
SFT_RUN = f"{MODEL.split('/')[-1]}-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500"
TEMPERATURE = 0.9
MAX_COMPLETION_LENGTH = 2048     # pass-1 cap (S was graded at 2048)
# Corruption data (judge): one normal completion per puzzle, then each kind's solution after corrupted reasoning
N_PUZZLES_4, N_PUZZLES_3, VAL_SHARE = 1000, 1000, 0.1
CUTS = (0.1, 0.25, 0.5)          # kinds: "empty" + "cut<f>" (first f of the reasoning's tokens, back to a whole line)
KEEP_UNFINISHED = True           # a corrupted solution without an answer stays as a rejected example, scored as written
TRAIN_KINDS = ("empty",)         # the OLMo judge's (validation: every kind)
MAX_SOLUTION_TOKENS = 128        # natural solutions over this, or without an answer, are not scored (the GRPO reward)
DPO_BETA, DPO_LR, DPO_EPOCHS = 0.1, 2e-5, 2
DPO_MAX_LENGTH = 640             # judge prompt + solution (+ answer)
# Calibration and the self-judge check: S's two-pass rollouts on the grade puzzles
CALIB_SAMPLES = 8
# GRPO arms (Consistency_GRPO_Olmo_Judge's recipe)
ARMS = ("format", "judge")       # run in this order; each skipped when its final adapter exists
JUDGE_REDUCTION = "sum"
MAX_STEPS = 150
PUZZLES_PER_STEP, NUM_GENERATIONS = 8, 16        # 4 four-number + 4 three-number puzzles x 16 = 128 per step
FORCE_ROOM = 128                 # pass 2: room for the forced solution + answer
LEARNING_RATE, WARMUP_STEPS = 2e-5, 10
KL_BETA = 0.02
LOSS_TYPE, SCALE_REWARDS, MAX_GRAD_NORM = "dapo", False, 0.2
VLLM_IS_MODE, VLLM_IS_CLIP = "token_truncate", 2.0
ADAM_BETA2, WEIGHT_DECAY = 0.99, 0.0
MICRO_BATCH = 8
EXPRESSION_REWARD = True
SAVE_STEPS, DUMP_EVERY = 10, 10
VLLM_GPU_MEMORY_UTILIZATION = 0.35
CURVE_EVERY, CURVE_N, CURVE_K = 25, 100, 4      # held-out curve: CURVE_N per size, greedy x 1 + sampled x CURVE_K
N_GRADE, GRADE_SAMPLES = 200, 8                  # S's grade puzzles per size, as S's grade file
GRADE_ALSO = (50, 100)                           # checkpoints graded besides the final policy
UNASSIGN_WHEN_DONE = True                        # [release]
# [prompt probe]: S's completions under alternative policy prompts (format block after the puzzle), for review
PROMPT_PROBE = True
PROBE_N, PROBE_ROOM = 100, 512                   # validation corruption puzzles per size; room after empty reasoning
_FMT = """

Respond in the following format:
<reasoning>
{your search for a way to reach the target}
</reasoning>
<solution>
{SOL}
</solution>
<answer>
{ANS}
</answer>"""
PROBE_PROMPTS = {
    "current": _FMT.replace("{SOL}", "{the steps, one per line, e.g. 25 * 4 = 100}")
                   .replace("{ANS}", "{one expression that uses each number exactly once and equals the target}"),
    "tried": _FMT.replace("{SOL}", "{the steps, one per line, e.g. 25 * 4 = 100; if you did not reach the target, the steps "
                                   "you tried}")
                 .replace("{ANS}", "{one expression that uses each number exactly once and equals the target, or none if "
                                   "you did not find one}"),
    "clear": _FMT.replace("{SOL}", "{a clear step-by-step solution, or a clear description of the steps you tried}")
                 .replace("{ANS}", "{one expression that uses each number exactly once and equals the target, or none}"),
}
if SMOKE:
    N_PUZZLES_4, N_PUZZLES_3, CALIB_SAMPLES, MAX_STEPS, SAVE_STEPS, DUMP_EVERY = 16, 16, 2, 2, 1, 1
    CURVE_EVERY, CURVE_N, CURVE_K, N_GRADE, GRADE_ALSO = 1, 4, 2, 4, (1,)

DRIVE = "/content/drive/MyDrive/Consistency"
CODE_DIR = f"{DRIVE}/code/countdown"
_sm = "-smoke" if SMOKE else ""
S_DIR = f"{DRIVE}/countdown/sft/{SFT_RUN}"
S_ADAPTER = f"{S_DIR}/adapter"
S_GRADE = f"{S_DIR}/grade_n200x8_t{TEMPERATURE:g}_cap{MAX_COMPLETION_LENGTH}_seed{SEED}/gen_sft.csv"
GRADE_FILES = {n: f"{DRIVE}/countdown/grpo/olmo3-7b-format50-cap1024-seed0/grade/gen_Olmo-3-7B-Instruct-format50_user_n{n}.csv"
               for n in (4, 3)}          # the grade puzzles (S's were drawn from these)
KINDS = ("empty",) + tuple(f"cut{c:g}" for c in CUTS)
CORRUPT_DIR = (f"{S_DIR}/judge_corrupt_{'-'.join(KINDS)}" + ("_unfinished" if KEEP_UNFINISHED else "")
               + f"_n4x{N_PUZZLES_4}_n3x{N_PUZZLES_3}_t{TEMPERATURE:g}_cap{MAX_COMPLETION_LENGTH}_seed{SEED}{_sm}")
JUDGE_DIR = f"{CORRUPT_DIR}/dpo_{'-'.join(TRAIN_KINDS)}_b{DPO_BETA:g}_lr{DPO_LR:g}_ep{DPO_EPOCHS}"
JUDGE_PATH = f"{JUDGE_DIR}/adapter"
CALIB_PATH = f"{JUDGE_DIR}/calibration_S_twopass_grade_x{CALIB_SAMPLES}_forced{FORCE_ROOM}.json"
JUDGE_TAG = "-".join(TRAIN_KINDS)
_short = MODEL_KEY + "-sft"
RUN_IDS = {arm: (f"{_short}-{arm}" + (f"-{JUDGE_REDUCTION}-{JUDGE_TAG}" if arm == "judge" else "")
                 + f"-forced{FORCE_ROOM}-n34-steps{MAX_STEPS}-cap{MAX_COMPLETION_LENGTH}-istt{VLLM_IS_CLIP:g}-seed{SEED}{_sm}")
           for arm in ARMS}
RUN_DIRS = {arm: f"{DRIVE}/countdown/grpo/{r}" for arm, r in RUN_IDS.items()}
CURVE_S = f"{S_DIR}/curve_S_n34x{CURVE_N}_greedy_pass{CURVE_N}x{CURVE_K}_t{TEMPERATURE:g}_cap{MAX_COMPLETION_LENGTH}_seed{SEED}{_sm}"
TRAIN_PUZZLES_CSV = f"{S_DIR}/grpo_train_puzzles_n34_steps{MAX_STEPS}x{PUZZLES_PER_STEP}_seed{SEED}{_sm}.csv"
assert set(ARMS) <= {"format", "judge"} and set(TRAIN_KINDS) <= set(KINDS) and JUDGE_REDUCTION in ("mean", "sum")
assert (PUZZLES_PER_STEP * NUM_GENERATIONS) % MICRO_BATCH == 0 and PUZZLES_PER_STEP % 2 == 0
assert all(s % SAVE_STEPS == 0 and s < MAX_STEPS for s in GRADE_ALSO)
print(f"MODEL={MODEL} {CHAT_KWARGS}\nS={S_ADAPTER}\nJUDGE_DIR={JUDGE_DIR}\n" + "\n".join(f"{a}: {d}" for a, d in RUN_DIRS.items()))


MODEL=Qwen/Qwen3-0.6B {'enable_thinking': False}
S=/content/drive/MyDrive/Consistency/countdown/sft/Qwen3-0.6B-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500/adapter
JUDGE_DIR=/content/drive/MyDrive/Consistency/countdown/sft/Qwen3-0.6B-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500/judge_corrupt_empty-cut0.1-cut0.25-cut0.5_unfinished_n4x1000_n3x1000_t0.9_cap2048_seed0/dpo_empty_b0.1_lr2e-05_ep2
format: /content/drive/MyDrive/Consistency/countdown/grpo/qwen3-0.6b-sft-format-forced128-n34-steps150-cap2048-istt2-seed0
judge: /content/drive/MyDrive/Consistency/countdown/grpo/qwen3-0.6b-sft-judge-sum-empty-forced128-n34-steps150-cap2048-istt2-seed0


In [ ]:
# [drive + HF] Mount Drive, HF login (Colab secret HF_TOKEN), and the checks that must fail before any GPU work.
import os, glob, shutil
from google.colab import drive, userdata
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
for f in ("countdown.py", "prompts.py", "rewards.py"):
    assert os.path.exists(f"{CODE_DIR}/{f}"), f"missing {CODE_DIR}/{f}"
assert os.path.exists(f"{S_ADAPTER}/adapter_model.safetensors"), f"no S adapter in {S_ADAPTER}"
for f in (S_GRADE, *GRADE_FILES.values()):
    assert os.path.exists(f), f"missing {f}"
for d in (CORRUPT_DIR, JUDGE_DIR, *RUN_DIRS.values()):
    os.makedirs(d, exist_ok=True)
print("judge:", "exists" if os.path.exists(f"{JUDGE_PATH}/adapter_model.safetensors") else "to train")
for a, d in RUN_DIRS.items():
    _c = glob.glob(f"{d}/checkpoint-*")
    print(f"{a}:", "done" if os.path.exists(f"{d}/adapter/adapter_model.safetensors")
          else (f"resumes from {max(_c, key=lambda p: int(p.rsplit('-', 1)[1]))}" if _c else "from S"))


Mounted at /content/drive
judge: to train
format: from S
judge: from S


In [ ]:
# [install] Once per fresh runtime, no restart needed (Consistency_GRPO_Olmo_Judge's): skipped when TRL >= 1 and vLLM are
# installed. pip runs as a subprocess, not %pip, so Colab shows no "restart the session" prompt; Pillow is pinned to the
# version Colab already imported. torchao is removed (peft refuses Colab's 0.10).
import importlib.metadata as _md, importlib.util, subprocess, sys
try:
    _ready = int(_md.version("trl").split(".")[0]) >= 1 and importlib.util.find_spec("vllm") is not None
except _md.PackageNotFoundError:
    _ready = False
if _ready:
    print(f"[install] skipped: trl {_md.version('trl')} and vllm {_md.version('vllm')} are installed")
else:
    _pin = [f"pillow=={_md.version('pillow')}"] if importlib.util.find_spec("PIL") else []
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "trl[vllm]", "peft", *_pin], check=True)
    print(f"[install] done: trl {_md.version('trl')}, vllm {_md.version('vllm')}" + (f", {_pin[0]} kept" if _pin else ""))
if importlib.util.find_spec("torchao"):
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("[install] removed torchao")


[install] done: trl 1.17.1, vllm 0.31.0, pillow==11.3.0 kept
[install] removed torchao


In [ ]:
# [vLLM setup] Before anything touches the GPU. (No expandable_segments: with it vLLM's CUDA-graph capture failed,
# Consistency_MATH_Finish_Probe, 2026-11-12.)
import os, sys, subprocess
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # keep the vLLM engine in this process
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"         # rollout_func is experimental in TRL
for stream, fd in ((sys.stdout, 1), (sys.stderr, 2)):   # Colab's stdout/stderr have no OS file descriptor
    try:
        stream.fileno()
    except Exception:
        stream.fileno = lambda fd=fd: fd
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
                     capture_output=True, text=True).stdout)
import importlib.metadata as md
for p in ["torch", "transformers", "trl", "peft", "datasets", "vllm"]:
    try:
        print(f"{p:14s}", md.version(p))
    except md.PackageNotFoundError:
        print(f"{p:14s}", "not installed")


name, memory.total [MiB], memory.used [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB, 0 MiB

torch          2.13.0
transformers   5.17.0
trl            1.17.1
peft           0.21.2
datasets       4.8.5
vllm           0.31.0


In [ ]:
# [setup] Tokenizer (CHAT_KWARGS on every template call), the Countdown code (Drive), the labels (the screen's / S's
# grade file's), the answer classes (corrupt-judge notebooks), and the judge's text views.
import re, json, random, importlib, math, gc, time
from collections import Counter
from fractions import Fraction
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
assert tok.pad_token is not None and tok.pad_token_id != tok.eos_token_id, tok.pad_token

def chat_text(messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **CHAT_KWARGS)

def chat_ids(messages):
    return tok(chat_text(messages), add_special_tokens=False)["input_ids"]

sys.path.insert(0, CODE_DIR)
import countdown, prompts, rewards as R
for _m in (countdown, prompts, R):
    importlib.reload(_m)
from countdown import check_answer
from prompts import countdown_prompt, judge_messages
tag, completion_text = R.tag, R.completion_text
_b = lambda d, c: d[c].astype(str) == "True"
_rd = lambda p: pd.read_csv(p, keep_default_na=False, dtype={c: str for c in ("completion", "reasoning", "solution", "answer",
                                                                              "generated", "text")})

def label(df):
    """As Consistency_SFT_Countdown's [grade] (the screen's): blocks, format_ok, answered, steps_parsed, correct, diagnosis."""
    df = df.copy()
    texts, nums, targets = list(df.completion), [json.loads(x) for x in df.numbers], list(df.target)
    for b in ("reasoning", "solution", "answer"):
        df[b] = [R.tag(t, b) or "" for t in texts]
    df["format_ok"] = [bool(re.search(r"<reasoning>.*?</reasoning>\s*<solution>.*?</solution>\s*<answer>.*?</answer>",
                                      t, re.DOTALL)) for t in texts]
    df["answered"] = df.answer.str.strip() != ""
    df["steps_parsed"] = [bool(R.parse_steps(s)) for s in df.solution]
    df["correct"] = np.array(R.correctness_reward_func(None, texts, nums, targets)) > 0
    df["diagnosis"] = [R.diagnose(a or None, n, t, tr) for a, n, t, tr in zip(df.answer, nums, targets, df.truncated)]
    return df

def read_graded(path):
    d = _rd(path)
    for c in ("finished", "truncated", "format_ok", "answered", "steps_parsed", "correct"):
        d[c] = _b(d, c)
    d["puzzle"], d["n_numbers"] = d.puzzle.astype(int), d.n_numbers.astype(int)
    return d

def stats(d):
    """As the SFT grade's table (cap MAX_COMPLETION_LENGTH)."""
    fin = d.finished
    ans, ok = fin & d.answered, fin & d.correct
    g = d.assign(aw=ans & ~ok, ok=ok).groupby("puzzle")
    return {"n": len(d), "accuracy": ok.mean(), "format ok": (fin & d.format_ok).mean(),
            "solution parses": (fin & d.steps_parsed).mean(), "truncated": 1 - fin.mean(),
            "answered wrong": (ans & ~ok).mean(), "correct among answered": ok.sum() / max(ans.sum(), 1),
            "tokens (median, finished)": d.n_tokens[fin].median(),
            "puzzles: correct + answered wrong": int((g.ok.any() & g.aw.any()).sum()), "pass@k": g.ok.any().mean()}

_NUM = r"\(?\s*(-?\d+(?:\.\d+)?(?:\s*/\s*\d+)?)\s*\)?"
_STEP = re.compile(rf"^\s*{_NUM}\s*([+\-*/×÷x])\s*{_NUM}\s*=\s*{_NUM}\s*\.?\s*$")

def replay(sol, numbers):
    """"false" / "unavailable" / ("valid", last result) / "unparsed" (the corrupt-judge notebooks')."""
    avail = Counter(Fraction(x) for x in numbers)
    bad = False
    lines = [l for l in (sol or "").splitlines() if l.strip()]
    if not lines:
        return "unparsed"
    for line in lines:
        m = _STEP.match(line.split("Left:")[0])
        if not m:
            return "unparsed"
        a, op, b, c = m.groups()
        a, b, c = (Fraction(s.replace(" ", "")) for s in (a, b, c))
        r = {"+": a + b, "-": a - b, "*": a * b, "×": a * b, "x": a * b}.get(op, a / b if b else None)
        if r != c:
            return "false"
        for v in (a, b):
            bad |= avail[v] < 1
            avail[v] -= 1
        avail[c] += 1
    return "unavailable" if bad else ("valid", c)

def klass(sol, ans, numbers, target):
    if not isinstance(ans, str) or not ans.strip():
        return "no answer"
    if check_answer(ans, list(numbers), int(target)):
        return "correct"
    r = replay(sol, numbers)
    if isinstance(r, tuple):
        return "bad answer" if r[1] == target else "honest miss"
    return {"false": "fake", "unavailable": "wrong numbers", "unparsed": "other"}[r]

def scorable(sol, ans):
    """Scored by the GRPO reward: a solution of at most MAX_SOLUTION_TOKENS tokens and an answer."""
    return (isinstance(sol, str) and bool(sol.strip()) and isinstance(ans, str) and bool(ans.strip())
            and len(tok(sol, add_special_tokens=False)["input_ids"]) <= MAX_SOLUTION_TOKENS)

judge_text = lambda sol, ans: f"{sol.strip()}\nAnswer: {ans.strip()}"

def unfinished_text(generated):
    """What was written inside <solution> without reaching an answer: up to </solution>, or all of it."""
    return generated.split("</solution>", 1)[0].strip()

def end_of_turn_id(tok):
    s = tok.apply_chat_template([{"role": "user", "content": "Q"}, {"role": "assistant", "content": "XYZXYZ"}],
                                tokenize=False, **CHAT_KWARGS)
    eot = tok(s.split("XYZXYZ", 1)[1], add_special_tokens=False)["input_ids"][0]
    assert tok.convert_ids_to_tokens(eot) in tok.all_special_tokens, tok.convert_ids_to_tokens(eot)
    return eot

EOT = end_of_turn_id(tok)
_p = chat_text([{"role": "user", "content": "x"}])
if CHAT_KWARGS.get("enable_thinking") is False:
    assert _p.rstrip().endswith("</think>"), "thinking-off template should end with an empty think block"
print("tokenizer:", MODEL, "| pad", tok.pad_token, "| eos", tok.eos_token, "| end of turn", tok.decode([EOT]),
      "| generation prompt ends", repr(_p[-40:]))


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

tokenizer: Qwen/Qwen3-0.6B | pad <|endoftext|> | eos <|im_end|> | end of turn <|im_end|> | generation prompt ends '|im_start|>assistant\n<think>\n\n</think>\n\n'


In [ ]:
# [puzzles] From Jiayi-Pan/Countdown-Tasks-3to4, disjoint by construction:
#   grade       S's grade puzzles (the grade files: 200 four- + 200 three-number of the test draw SEED), first N_GRADE
#   corruption  draw SEED + 6 (the OLMo / Llama judges' draw), apart from the test, SFT (SEED + 1) and Llama GRPO draws
#   training    draw SEED + 8: MAX_STEPS x PUZZLES_PER_STEP / 2 of each size, per step 4 four- then 4 three-number,
#               apart from all of the above (whole draws) and the corruption draw; saved to TRAIN_PUZZLES_CSV (both arms)
#   curve       draw SEED + 10: CURVE_N of each size, apart from everything above
_tz = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", split="train")
_as = lambda r: {"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])}
_key = lambda p: (tuple(sorted(p["numbers"])), p["target"])
_seeded = lambda seed, size: _tz.select(np.random.default_rng(seed).choice(len(_tz), size=min(len(_tz), size), replace=False))

GRADE = {}
for n, f in GRADE_FILES.items():
    g = pd.read_csv(f, keep_default_na=False).drop_duplicates("puzzle").sort_values("puzzle").head(N_GRADE)
    GRADE[n] = [{"puzzle": int(r.puzzle), "numbers": json.loads(r.numbers), "target": int(r.target)} for r in g.itertuples()]
_draw0 = _seeded(SEED, 20 * 200)
assert [_key(p) for p in GRADE[4]] == [_key(_as(r)) for r in _draw0 if len(r["nums"]) == 4][1:N_GRADE + 1]

_no = {_key(_as(r)) for seed, size in ((SEED, 20 * 200), (SEED + 1, 6 * (4000 + 100)), (SEED + 2, 4 * (4000 + 100)),
                                        (SEED + 3, 4 * 3900)) for r in _seeded(seed, size)}
_want, _got = {4: N_PUZZLES_4, 3: N_PUZZLES_3}, {4: [], 3: []}
for r in _seeded(SEED + 6, 10 * (N_PUZZLES_4 + N_PUZZLES_3)):
    p = _as(r)
    n = len(p["numbers"])
    if _key(p) not in _no and len(_got[n]) < _want[n]:
        _no.add(_key(p))
        _got[n].append(p)
assert all(len(_got[n]) == _want[n] for n in _want), {n: len(v) for n, v in _got.items()}
corrupt_puzzles = []
for n in (4, 3):
    n_val = round(VAL_SHARE * _want[n])
    corrupt_puzzles += [{**p, "pid": len(corrupt_puzzles) + i, "n": n, "val": i >= _want[n] - n_val} for i, p in enumerate(_got[n])]
_no |= {_key(_as(r)) for r in _seeded(SEED + 6, 10 * (N_PUZZLES_4 + N_PUZZLES_3))}

_half = PUZZLES_PER_STEP // 2
_need = MAX_STEPS * _half
_tr = {4: [], 3: []}
for r in _seeded(SEED + 8, 40 * _need):
    p = _as(r)
    n = len(p["numbers"])
    if _key(p) not in _no and len(_tr[n]) < _need:
        _no.add(_key(p))
        _tr[n].append(p)
assert all(len(v) == _need for v in _tr.values()), {n: len(v) for n, v in _tr.items()}
_train = [p for s in range(MAX_STEPS) for n in (4, 3) for p in _tr[n][s * _half:(s + 1) * _half]]
_tdf = pd.DataFrame([{"pid": i, "n": len(p["numbers"]), "numbers": json.dumps(p["numbers"]), "target": p["target"]}
                     for i, p in enumerate(_train)])
if os.path.exists(TRAIN_PUZZLES_CSV):
    assert pd.read_csv(TRAIN_PUZZLES_CSV)[["numbers", "target"]].astype(str).equals(_tdf[["numbers", "target"]].astype(str)), \
        f"{TRAIN_PUZZLES_CSV} differs from this draw"
else:
    _tdf.to_csv(TRAIN_PUZZLES_CSV, index=False)
train_dataset = Dataset.from_list([{"prompt": countdown_prompt(p["numbers"], p["target"]), "numbers": p["numbers"],
                                    "target": p["target"], "pid": i, "n": len(p["numbers"])} for i, p in enumerate(_train)])
CURVE = {4: [], 3: []}
for r in _seeded(SEED + 10, 40 * CURVE_N):
    p = _as(r)
    n = len(p["numbers"])
    if _key(p) not in _no and len(CURVE[n]) < CURVE_N:
        _no.add(_key(p))
        CURVE[n].append(p)
assert all(len(v) == CURVE_N for v in CURVE.values())
_gk = {_key(p) for v in GRADE.values() for p in v}
assert not _gk & ({_key(p) for p in _train} | {_key(p) for p in corrupt_puzzles} | {_key(p) for v in CURVE.values() for p in v})
print(f"grade {({n: len(v) for n, v in GRADE.items()})}, corruption {len(corrupt_puzzles)} (validation "
      f"{sum(p['val'] for p in corrupt_puzzles)}), training {len(train_dataset)} ({MAX_STEPS} steps x {_half} + {_half}), "
      f"curve {({n: len(v) for n, v in CURVE.items()})}")


README.md:   0%|          | 0.00/314 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.85MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/490364 [00:00<?, ? examples/s]

grade {4: 200, 3: 200}, corruption 2000 (validation 200), training 1200 (150 steps x 4 + 4), curve {4: 100, 3: 100}


In [ ]:
# [vLLM sync fix] (Consistency_GRPO_Olmo_Judge's) TRL syncs LoRA to vLLM by merging dW into the training model's bf16 W
# in place, then unmerging; each add/subtract rounds, so the frozen W drifts. This version streams bf16(W + dW) built in
# a temporary tensor (one rounding) and never touches the training model. Only active adapters are merged.
import torch
from accelerate.utils import is_peft_model
from peft.tuners.lora import LoraLayer
from trl.generation.vllm_generation import VLLMGeneration

if not hasattr(VLLMGeneration, "_trl_iter_named_params"):
    VLLMGeneration._trl_iter_named_params = VLLMGeneration._iter_named_params

def _iter_named_params_out_of_place(self):
    model = self.model
    if not is_peft_model(model) or self.accelerator.num_processes > 1:
        yield from self._trl_iter_named_params()
        return
    lora_layers = {name: m for name, m in model.named_modules() if isinstance(m, LoraLayer)}
    n_merged = 0
    for name, param in model.named_parameters():
        vllm_name = name.removeprefix("base_model.model.").replace(".base_layer", "")
        if model.prefix in vllm_name or "original_module" in vllm_name:
            continue
        vllm_name = self._fix_param_name_to_vllm(vllm_name, extra_prefixes=["modules_to_save.default."])
        layer = lora_layers.get(name.rsplit(".base_layer.", 1)[0]) if name.endswith(".base_layer.weight") else None
        if layer is None:
            yield vllm_name, param.data
            continue
        assert not layer.merged, f"{name}: adapter is merged in the training model"
        with torch.no_grad():
            merged = param.data.float()
            for adapter in layer.active_adapters:
                if adapter in layer.lora_A:
                    merged += layer.get_delta_weight(adapter).float()
            merged = merged.to(param.dtype)
        n_merged += 1
        yield vllm_name, merged
        del merged
    if not getattr(self, "_out_of_place_reported", False):
        print(f"[vLLM sync fix] streamed {n_merged} LoRA layers as bf16(W + dW); training weights untouched")
        self._out_of_place_reported = True

VLLMGeneration._iter_named_params = _iter_named_params_out_of_place
print("VLLMGeneration._iter_named_params patched")


VLLMGeneration._iter_named_params patched


In [ ]:
# [rollout] Consistency_GRPO_Olmo_Judge's two-pass rollouts (TRL's rollout_func, both arms). Pass 1 samples to the cap. A
# truncated one that never reached <solution> is cut back to its last whole line (or last word if that drops over 20%),
# "</reasoning>\n<solution>\n" is appended, and pass 2 writes the solution and answer in FORCE_ROOM tokens (stop at
# </answer>). The appended tags get env_mask 0 (no loss, no IS ratio). Extra fields: natural, forced, forced_gen.
from vllm import SamplingParams

def cut_point(ids):
    pieces = [tok.decode([i]) for i in ids]
    ends = np.cumsum([len(p) for p in pieces]) if pieces else np.array([0])
    line = [j + 1 for j, p in enumerate(pieces) if p.endswith("\n")] + [j for j, p in enumerate(pieces) if p.startswith("\n")]
    k = max(line, default=0)
    if k == 0 or ends[k - 1] < 0.8 * ends[-1]:
        k = max([j for j, p in enumerate(pieces) if p.startswith(" ")], default=k)
    while k > 0 and not pieces[k - 1].strip():
        k -= 1
    return k

def forced_tags(head):
    tags = "<solution>\n" if "</reasoning>" in head else "</reasoning>\n<solution>\n"
    return tags if head.endswith("\n") else "\n" + tags

_EOS = {tok.eos_token_id, EOT, tok.pad_token_id}
ROLLOUT_STATS = {}

def two_pass_rollout(prompts, trainer):
    prompt_ids = trainer._tokenize_prompts(prompts)[0]
    _, ids1, lps1, _, _ = trainer.vllm_generation.generate(prompts=prompt_ids, images=None,
                                                           num_generations=trainer.num_generations)
    ids1 = [list(x) for x in ids1]
    lps1 = [[lp[0] for lp in seq] for seq in lps1]
    natural = [tok.decode(x, skip_special_tokens=True) for x in ids1]
    truncated = [len(x) >= MAX_COMPLETION_LENGTH and x[-1] not in _EOS for x in ids1]
    todo = [i for i, (t, tr) in enumerate(zip(natural, truncated)) if tr and "<solution>" not in t]
    heads, tags = {}, {}
    for i in todo:
        heads[i] = cut_point(ids1[i])
        tags[i] = tok(forced_tags(tok.decode(ids1[i][:heads[i]])), add_special_tokens=False)["input_ids"]
    outs = trainer.vllm_generation.llm.generate(
        [{"prompt_token_ids": prompt_ids[i] + ids1[i][:heads[i]] + tags[i]} for i in todo],
        SamplingParams(n=1, temperature=TEMPERATURE, top_p=1.0, max_tokens=FORCE_ROOM, stop=["</answer>"],
                       include_stop_str_in_output=True, logprobs=0), use_tqdm=False) if todo else []
    comp, lps, mask, forced_gen = [list(x) for x in ids1], [list(x) for x in lps1], [[1] * len(x) for x in ids1], [""] * len(ids1)
    for i, o in zip(todo, outs):
        out = o.outputs[0]
        g = list(out.token_ids)
        glp = [d[t].logprob for d, t in zip(out.logprobs, g)]
        k = heads[i]
        comp[i] = ids1[i][:k] + tags[i] + g
        lps[i] = lps1[i][:k] + [0.0] * len(tags[i]) + glp
        mask[i] = [1] * k + [0] * len(tags[i]) + [1] * len(g)
        forced_gen[i] = out.text
    forced = [i in heads for i in range(len(ids1))]
    ROLLOUT_STATS.update(n=len(ids1), truncated=float(np.mean(truncated)), forced=float(np.mean(forced)),
                         forced_answered=float(np.mean([g.rstrip().endswith("</answer>") for g in forced_gen if g] or [np.nan])),
                         truncated_in_solution=float(np.mean([tr and i not in heads for i, tr in enumerate(truncated)])))
    for k_, v in ROLLOUT_STATS.items():
        if k_ != "n" and not math.isnan(v):
            trainer._metrics["train"][f"rollout/{k_}"].append(v)
    return {"prompt_ids": prompt_ids, "completion_ids": comp, "logprobs": lps, "env_mask": mask,
            "natural": natural, "forced": forced, "forced_gen": forced_gen}

print("two_pass_rollout defined; e.g. forced tags:", repr(forced_tags("Try 5 * 4 = 20")))


two_pass_rollout defined; e.g. forced tags: '\n</reasoning>\n<solution>\n'


In [ ]:
# [rewards] Consistency_GRPO_Olmo_Judge's: S's format rewards (xmlcount, soft / strict format, expression) on the NATURAL
# (pass-1) text; the judge (reward weight set per arm in [train]: calibrated in the judge arm, 0 in the format arm)
# scores, under its frozen adapter, the judge prompt + the text + end of turn:
#   forced     the forced solution + answer, or unfinished: what pass 2 wrote inside <solution>
#   truncated inside <solution> (not forced): what was written inside <solution>
#   finished   solution + answer if both parse and the solution has <= MAX_SOLUTION_TOKENS tokens, else missing
# Missing = JUDGE_MISSING (set by [calibrate]). Logged with weight 0: correctness (natural, final), forced, the judge's
# mean and sum, S's monitors; the step monitor prints every step and dumps every DUMP_EVERY to the arm's RUN_DIR.
import contextlib
from trl.trainer.utils import selective_log_softmax
CUR = {"arm": None, "run_dir": None}            # the arm [train] runs
JUDGE_MISSING = float("nan")                    # set by [calibrate]

def on_natural(f):
    def g(completions, natural=None, **kwargs):
        cs = completions if natural is None else [[{"role": "assistant", "content": t}] for t in natural]
        return f(completions=cs, **kwargs)
    g.__name__ = f.__name__
    return g

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>") == 1:
        count += 0.125
    if text.count("</reasoning>") == 1:
        count += 0.125
    if text.count("<solution>") == 1:
        count += 0.005
    if text.count("</solution>") == 1:
        count += 0.005
    if text.count("<answer>") == 1:
        count += 0.125
    if text.count("</answer>") == 1:
        count += 0.125
    if "</answer>" in text:
        count -= (len(text.split("</answer>", 1)[1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    return [count_xml(completion_text(c)) for c in completions]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"<reasoning>[\s\S]*?</reasoning>\s*<solution>[\s\S]*?</solution>\s*<answer>[\s\S]*?</answer>"
    return [0.5 if re.match(pattern, completion_text(c)) else 0.0 for c in completions]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"<reasoning>\s*.*?\s*</reasoning>\s*<solution>\s*.*?\s*</solution>\s*<answer>\s*.*?\s*</answer>"
    return [0.5 if re.search(pattern, completion_text(c), re.DOTALL) else 0.0 for c in completions]

FORMAT_FUNCS = [on_natural(f) for f in [xmlcount_reward_func, soft_format_reward_func, strict_format_reward_func]
                + ([R.expression_reward_func] if EXPRESSION_REWARD else [])]

def format_total(natural):
    """The summed format reward of pass-1 texts (what the format arm trains on)."""
    cs = [[{"role": "assistant", "content": t}] for t in natural]
    return np.sum([f(completions=cs) for f in FORMAT_FUNCS], axis=0)

def judge_input(natural, forced, forced_gen):
    """The text the judge scores, or None (missing)."""
    if forced:
        s, a = tag("<solution>\n" + forced_gen, "solution"), tag("<solution>\n" + forced_gen, "answer")
        return judge_text(s, a) if (s and a) else (unfinished_text(forced_gen) or None)
    s, a = tag(natural, "solution"), tag(natural, "answer")
    if s and a:
        return judge_text(s, a) if len(tok(s, add_special_tokens=False)["input_ids"]) <= MAX_SOLUTION_TOKENS else None
    if "<solution>" in natural and "</answer>" not in natural:
        return unfinished_text(natural.split("<solution>", 1)[1]) or None
    return None

def _freeze_others(model):
    for n, p in model.named_parameters():        # set_adapter makes the active adapter trainable: only the policy is
        if "lora_" in n and ".default." not in n:
            p.requires_grad_(False)

@contextlib.contextmanager
def scorer_weights(model, adapter):
    """adapter: "judge", "ref" (= S, frozen) or None (the untrained model: adapters off); then the policy again."""
    if adapter is None:
        with model.disable_adapter():
            yield
        return
    model.set_adapter(adapter)
    try:
        yield
    finally:
        model.set_adapter("default")
        _freeze_others(model)

def judge_scores(model, items, adapter="judge", chunk=16):
    """(mean, sum) arrays over items = [(numbers, target, text or None)], NaN for None: per-token log-probs of the text +
    end of turn after the judge prompt (Consistency_Corrupt_Judge_Olmo's score)."""
    means, sums = np.full(len(items), np.nan), np.full(len(items), np.nan)
    seqs = {i: (chat_ids(judge_messages(list(n), int(t))), tok(x, add_special_tokens=False)["input_ids"] + [EOT])
            for i, (n, t, x) in enumerate(items) if x}
    order = sorted(seqs, key=lambda i: len(seqs[i][0]) + len(seqs[i][1]))
    fwd = getattr(model, "_original_forward", model.forward)
    with torch.no_grad(), scorer_weights(model, adapter):
        for s in range(0, len(order), chunk):
            idx = order[s:s + chunk]
            L = max(len(seqs[i][0]) + len(seqs[i][1]) for i in idx)
            ids = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long)
            attn = torch.zeros_like(ids)
            for j, i in enumerate(idx):
                x = seqs[i][0] + seqs[i][1]
                ids[j, :len(x)], attn[j, :len(x)] = torch.tensor(x), 1
            ids, attn = ids.to(model.device), attn.to(model.device)
            per_pos = selective_log_softmax(fwd(input_ids=ids, attention_mask=attn).logits[:, :-1], ids[:, 1:]).float()
            for j, i in enumerate(idx):
                p, t = seqs[i]
                lp = per_pos[j, len(p) - 1:len(p) + len(t) - 1].cpu().numpy()
                means[i], sums[i] = lp.mean(), lp.sum()
            del per_pos
    return means, sums

_LAST = {}

def make_judge_reward(owner):
    def judge(completions, numbers, target, natural, forced, forced_gen, **kwargs):
        assert "judge" in owner.model.peft_config and not math.isnan(JUDGE_MISSING), "run [judge DPO] and [calibrate] first"
        items = [(n, t, judge_input(x, f, g)) for n, t, x, f, g in zip(numbers, target, natural, forced, forced_gen)]
        m, s = judge_scores(owner.model, items)
        _LAST.update(id=id(completions), mean=m, sum=s)
        v = m if JUDGE_REDUCTION == "mean" else s
        return [JUDGE_MISSING if math.isnan(x) else float(x) for x in v]
    judge.__name__ = "judge"
    return judge

def _last(key, completions):
    v = _LAST.get(key) if _LAST.get("id") == id(completions) else None
    return [None] * len(completions) if v is None else [None if math.isnan(x) else float(x) for x in v]

def judge_mean(completions, **kwargs):
    return _last("mean", completions)

def judge_sum(completions, **kwargs):
    return _last("sum", completions)

def correct_natural(completions, numbers, target, natural, **kwargs):
    return [float(x > 0) for x in R.correctness_reward_func(None, natural, numbers, target)]

def correct_final(completions, numbers, target, **kwargs):
    return [float(x > 0) for x in R.correctness_reward_func(None, completions, numbers, target)]

def forced_flag(completions, forced, **kwargs):
    return [float(f) for f in forced]

MONITORS = [on_natural(f) for f in R.MONITORS]

def within_puzzle_win_rate(keys, scores, correct):
    """Over (correct, wrong) pairs of the same puzzle: share where the correct one scores higher, ties 1/2."""
    wins = pairs = 0
    keys = np.array(keys)
    for k in set(keys.tolist()):
        s, c = scores[keys == k], correct[keys == k]
        d = s[c][:, None] - s[~c][None, :]
        wins += (d > 0).sum() + 0.5 * (d == 0).sum()
        pairs += d.size
    return (wins / pairs if pairs else float("nan")), int(pairs)

def make_step_monitor(owner):
    def step_monitor(prompts, completions, numbers, target, pid, natural, forced, forced_gen, **kwargs):
        step_monitor_check(completions)
        cn = np.array(correct_natural(completions, numbers, target, natural)) > 0
        cf = np.array(correct_final(completions, numbers, target)) > 0
        fz = np.array(forced)
        fin = np.array([("</answer>" in t) for t in natural])
        v = np.array(judge_mean(completions) if JUDGE_REDUCTION == "mean" else judge_sum(completions), dtype=float)
        v = np.where(np.isnan(v), JUDGE_MISSING, v)
        rate, pairs = within_puzzle_win_rate(pid, v, cf)
        n_ = np.array([len(x) for x in numbers])
        line = (f"correct natural {cn.mean():.3f} (4: {cn[n_ == 4].mean():.3f}, 3: {cn[n_ == 3].mean():.3f}) final "
                f"{cf.mean():.3f} | forced {fz.mean():.2f} | correct among finished "
                f"{cn[fin].mean() if fin.any() else float('nan'):.3f} | judge {JUDGE_REDUCTION} {v.mean():.3f}, "
                f"win {rate:.2f} ({pairs} pairs)")
        log = kwargs.get("log_metric")
        if log:
            log("correct_among_finished", float(cn[fin].mean()) if fin.any() else float("nan"))
            if pairs:
                log("judge_win_rate", float(rate))
        step = owner.state.global_step
        print(f"[{CUR['arm']} step {step}] {line}")
        if step % DUMP_EVERY == 0:
            with open(f"{CUR['run_dir']}/step_{step:05d}.txt", "w", encoding="utf-8") as f:
                f.write(f"=== Step {step} ===\n\n")
                for i, p in enumerate(prompts):
                    f.write(f"### Sample {i + 1}\n[Scores] puzzle={pid[i]} correct_natural={bool(cn[i])} "
                            f"correct_final={bool(cf[i])} forced={bool(fz[i])} judge_{JUDGE_REDUCTION}={v[i]:.4f}\n"
                            f"[Prompt]\n{p[-1]['content'].strip()}\n\n[Response]\n{natural[i].strip()}\n\n"
                            + (f"[Forced]\n{forced_gen[i].strip()}\n\n" if fz[i] else "") + "=" * 80 + "\n\n")
        return [0.0] * len(completions)
    step_monitor.__name__ = "step_monitor"
    return step_monitor

def step_monitor_check(completions):
    """Qwen3 thinking off: no training completion may open <think> (TRL's chat_template_kwargs applied)."""
    if CHAT_KWARGS.get("enable_thinking") is False:
        _t = sum("<think>" in completion_text(c) for c in completions)
        assert _t == 0, f"{_t} training completions open <think>: TRL did not apply enable_thinking=False"
    return True

REWARD_FUNCS_TRAIN = ([(f, 1.0) for f in FORMAT_FUNCS] + [("judge", 0.0)]
                      + [(correct_natural, 0.0), (correct_final, 0.0), (forced_flag, 0.0), (judge_mean, 0.0),
                         (judge_sum, 0.0)] + [(f, 0.0) for f in MONITORS] + [("step_monitor", 0.0)])
JUDGE_IDX = [f for f, _ in REWARD_FUNCS_TRAIN].index("judge")
print("reward functions:", [f if isinstance(f, str) else f.__name__ for f, _ in REWARD_FUNCS_TRAIN])


reward functions: ['xmlcount_reward_func', 'soft_format_reward_func', 'strict_format_reward_func', 'expression_reward_func', 'judge', 'correct_natural', 'correct_final', 'forced_flag', 'judge_mean', 'judge_sum', 'uses_given_numbers', 'made_up_numbers', 'solution_arithmetic_ok', 'solution_reaches_target', 'answer_matches_solution', 'stopped', 'gave_up', 'caught_claims', 'step_monitor']


In [ ]:
# [build] The policy (S's adapter "default", trainable), the GRPO config and trainer with the two-pass rollout (TRL copies
# S to a frozen "ref" adapter: the KL reference, and S's own scorer in [calibrate]); its vLLM engine samples everything in
# this notebook. Checks: adapters, the trainer's prompt tokens = chat_ids (thinking off), vLLM vs trainer log-probs, a
# real two-pass batch. The judge adapter is added after [judge DPO].
import csv, tempfile, inspect
from datetime import datetime
from transformers import AutoModelForCausalLM, TrainerCallback
from peft import PeftModel
from trl import GRPOTrainer, GRPOConfig
assert "rollout_func" in inspect.signature(GRPOTrainer.__init__).parameters, "this TRL has no rollout_func: rerun [install]"

class CSVLoggerCallback(TrainerCallback):
    """Every log row to a CSV; the header grows when new keys appear."""
    def __init__(self, path):
        self.path, self.fields = path, None
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if os.path.exists(path) and os.path.getsize(path):
            with open(path, newline="") as f:
                self.fields = next(csv.reader(f))
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        logs = {"step": state.global_step, **logs}
        if self.fields is None:
            self.fields = list(logs)
            with open(self.path, "w", newline="") as f:
                csv.DictWriter(f, fieldnames=self.fields).writeheader()
        new = [k for k in logs if k not in self.fields]
        if new:
            self.fields += sorted(new)
            rows = list(csv.DictReader(open(self.path, newline="")))
            fd, tmp = tempfile.mkstemp(dir=os.path.dirname(self.path))
            os.close(fd)
            with open(tmp, "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=self.fields)
                w.writeheader()
                w.writerows(rows)
            shutil.move(tmp, self.path)
        with open(self.path, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=self.fields).writerow(logs)

class JudgeGRPOTrainer(GRPOTrainer):
    def log(self, logs, start_time=None):
        super().log(logs, start_time)
        if "reward" in logs and "train_runtime" not in logs:
            v = lambda k: float("nan") if logs.get(k) is None else float(logs[k])
            print(f"[{CUR['arm']} {self.state.global_step}] reward={v('reward'):.3f}, no-gradient groups="
                  f"{v('frac_reward_zero_std'):.2f}, grad_norm={v('grad_norm'):.3f}, kl={v('kl'):.4f}, "
                  f"length={v('completions/mean_length'):.0f}, truncated={v('rollout/truncated'):.3f}, "
                  f"correct natural={v('rewards/correct_natural/mean'):.3f}, judge {JUDGE_REDUCTION}="
                  f"{v(f'rewards/judge_{JUDGE_REDUCTION}/mean'):.3f}")

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="auto")
model.config.pad_token_id = tok.pad_token_id
model.generation_config.pad_token_id = tok.pad_token_id
model = PeftModel.from_pretrained(model, S_ADAPTER, is_trainable=True)          # "default" = S, continued per arm

COMPLETIONS_PER_STEP = PUZZLES_PER_STEP * NUM_GENERATIONS
training_args = GRPOConfig(
    use_vllm=True, vllm_mode="colocate", vllm_gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    vllm_max_model_length=MAX_COMPLETION_LENGTH + FORCE_ROOM + 512,
    chat_template_kwargs=CHAT_KWARGS, gradient_checkpointing=True,
    loss_type=LOSS_TYPE, scale_rewards=SCALE_REWARDS, max_grad_norm=MAX_GRAD_NORM, temperature=TEMPERATURE,
    beta=KL_BETA, reward_weights=[w for _, w in REWARD_FUNCS_TRAIN],
    learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS, lr_scheduler_type="constant_with_warmup", optim="adamw_torch",
    adam_beta1=0.9, adam_beta2=ADAM_BETA2, weight_decay=WEIGHT_DECAY,
    vllm_importance_sampling_correction=True, vllm_importance_sampling_mode=VLLM_IS_MODE,
    vllm_importance_sampling_clip_max=VLLM_IS_CLIP,
    seed=SEED, shuffle_dataset=False, logging_strategy="steps", logging_steps=5,
    save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=None, output_dir=f"{JUDGE_DIR}/trainer_unused", report_to=[],
    bf16=True, per_device_train_batch_size=MICRO_BATCH, gradient_accumulation_steps=COMPLETIONS_PER_STEP // MICRO_BATCH,
    num_generations=NUM_GENERATIONS, max_completion_length=MAX_COMPLETION_LENGTH, max_steps=MAX_STEPS,
)
_bound = {}

def _late(name):
    def f(**kwargs):
        return _bound[name](**kwargs)
    f.__name__ = name
    return f

trainer = JudgeGRPOTrainer(model=model, processing_class=tok, args=training_args, train_dataset=train_dataset,
                           reward_funcs=[_late(f) if isinstance(f, str) else f for f, _ in REWARD_FUNCS_TRAIN],
                           rollout_func=two_pass_rollout)
_bound.update(judge=make_judge_reward(trainer), step_monitor=make_step_monitor(trainer))
_pm = trainer.model
assert {"default", "ref"} <= set(_pm.peft_config), f"adapters: {list(_pm.peft_config)}"
assert all(p.requires_grad == (".default." in n) for n, p in _pm.named_parameters() if "lora_" in n), "a non-policy adapter trains"
_d = {n: p for n, p in _pm.named_parameters() if ".default." in n and "lora_" in n}
assert all(torch.equal(p, _pm.get_parameter(n.replace(".default.", ".ref."))) for n, p in _d.items()), "ref != S"
assert trainer._tokenize_prompts([train_dataset[0]["prompt"]])[0][0] == chat_ids(train_dataset[0]["prompt"]), \
    "the trainer's prompt tokens differ from chat_ids (chat_template_kwargs?)"
print(f"[adapters] {sorted(_pm.peft_config)}; policy trainable ({sum(p.numel() for p in _d.values()):,} params), ref = S; "
      "trainer prompt tokens = chat_ids")

trainer.vllm_generation.sync_weights()
POLICY = "S"
_n = 0
for m in trainer.model.modules():
    if getattr(type(m).__call__, "__module__", "").startswith("vllm") and not hasattr(m, "do_not_compile"):
        m.do_not_compile = True
        _n += 1
print(f"[vLLM patch] {_n} module(s) marked do_not_compile")

def load_policy(path):
    """An adapter's weights into "default" (the policy), synced to vLLM."""
    from safetensors.torch import load_file
    from peft import set_peft_model_state_dict
    res = set_peft_model_state_dict(trainer.model, load_file(f"{path}/adapter_model.safetensors"), adapter_name="default")
    assert not res.unexpected_keys, res.unexpected_keys[:4]
    trainer.vllm_generation.sync_weights()

def trainer_logps(ids_prompt, ids_completion, temperature):
    inp = torch.tensor([ids_prompt + ids_completion], device=trainer.model.device)
    with torch.no_grad():
        logits = trainer.model(input_ids=inp).logits[0, len(ids_prompt) - 1:-1].float() / temperature
    return torch.log_softmax(logits, -1).gather(1, inp[0, len(ids_prompt):, None])[:, 0].cpu().numpy()

SAMPLER_MAX_DIFF = 0.02
_ids = [chat_ids(countdown_prompt(p["numbers"], p["target"])) for p in GRADE[4][:8]]
_outs = trainer.vllm_generation.llm.generate([{"prompt_token_ids": x} for x in _ids],
                                             SamplingParams(temperature=0.0, max_tokens=128, logprobs=1), use_tqdm=False)
_dd = []
for x, o in zip(_ids, _outs):
    t = list(o.outputs[0].token_ids)
    _dd.append(trainer_logps(x, t, 1.0) - np.array([lp[i].logprob for lp, i in zip(o.outputs[0].logprobs, t)]))
_seq = np.array([s.sum() for s in _dd])
_dd = np.concatenate(_dd)
print(f"[sampler check] {len(_dd)} tokens: mean |diff| {np.abs(_dd).mean():.4f}, signed {_dd.mean():+.4f}; "
      f"per-sequence log-ratio {np.round(_seq, 3).tolist()}")
assert np.abs(_dd).mean() < SAMPLER_MAX_DIFF and abs(_dd.mean()) < 0.01, "vLLM and the trainer disagree"

_rows = [train_dataset[i] for i in range(2) for _ in range(NUM_GENERATIONS)]
_ro = two_pass_rollout([r["prompt"] for r in _rows], trainer)
assert all(len(c) == len(l) == len(m) for c, l, m in zip(_ro["completion_ids"], _ro["logprobs"], _ro["env_mask"]))
assert not any("<think>" in t for t in _ro["natural"]), "a rollout opened <think>"
_f = [i for i, f in enumerate(_ro["forced"]) if f]
print(f"[two-pass check] {len(_rows)} rollouts: {ROLLOUT_STATS}")
for i in _f[:3]:
    c, m = np.array(_ro["completion_ids"][i]), np.array(_ro["env_mask"][i])
    d = (trainer_logps(_ro["prompt_ids"][i], list(c), TEMPERATURE) - np.array(_ro["logprobs"][i]))[m == 1]
    print(f"  forced rollout {i}: mean |diff| on policy tokens {np.abs(d).mean():.4f}")
    assert np.abs(d).mean() < SAMPLER_MAX_DIFF, "forced rollout: vLLM and trainer log-probs disagree"
trainer._metrics["train"].clear()
print("[build] trainer ready")


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Capturing CUDA graphs (FULL): 100%|██████████| 19/19 [00:00<00:00, 25.90it/s]


[adapters] ['default', 'ref']; policy trainable (10,092,544 params), ref = S; trainer prompt tokens = chat_ids
[vLLM sync fix] streamed 196 LoRA layers as bf16(W + dW); training weights untouched
[vLLM patch] 0 module(s) marked do_not_compile
[sampler check] 1024 tokens: mean |diff| 0.0026, signed +0.0001; per-sequence log-ratio [-0.068, 0.284, 0.071, -0.061, -0.138, 0.075, -0.067, -0.015]
[two-pass check] 32 rollouts: {'n': 32, 'truncated': 0.6875, 'forced': 0.65625, 'forced_answered': 0.5714285714285714, 'truncated_in_solution': 0.03125}
  forced rollout 0: mean |diff| on policy tokens 0.0039
  forced rollout 1: mean |diff| on policy tokens 0.0032
  forced rollout 3: mean |diff| on policy tokens 0.0031
[build] trainer ready


In [ ]:
# [prompt probe] PROBE_PROMPTS (in [config]) on part 1's validation corruption puzzles (PROBE_N per size), S in the engine:
#   normal  S's completion under the prompt (T, cap, as [corruption])
#   empty   S's solution + answer after empty reasoning ("<reasoning>\n</reasoning>\n<solution>\n", as the judge's
#           corruption), up to PROBE_ROOM tokens; [corruption] gave it ROOM (corrupt.csv's budget) tokens
# Checks (measurement only): cut off (normal: hit the cap; empty: no </answer> within ROOM / PROBE_ROOM), answered,
# answer "none", answer classes, solution tokens (> MAX_SOLUTION_TOKENS = unscored in GRPO), the judge's sum and mean by
# class. Five examples per prompt and side (one per class in turn). Saved to PROBE_DIR (+ review.txt with the examples).
PROBE_DIR = f"{CORRUPT_DIR}/prompt_probe_n{PROBE_N}_seed{SEED}"
os.makedirs(PROBE_DIR, exist_ok=True)
ROOM = int(corrupt.budget.iloc[0])
_NONE = re.compile(r"\s*(none|no solution|not found|n/?a)\b.*", re.I | re.S)
_pp = [p for n in (4, 3) for p in [q for q in corrupt_puzzles if q["n"] == n and q["val"]][:PROBE_N]]
_ntok = lambda s: len(tok(s, add_special_tokens=False)["input_ids"]) if isinstance(s, str) and s else 0

def _probe_prompt(p, suffix):
    return [{"role": "user", "content": prompts.countdown_question(p["numbers"], p["target"]) + suffix}]

def _row(p, name, side, full, gen_tokens, cut_off):
    sol, ans = tag(full, "solution"), tag(full, "answer")
    return {"prompt": name, "side": side, "pid": p["pid"], "n": p["n"], "numbers": json.dumps(p["numbers"]),
            "target": p["target"], "text": full, "tokens": gen_tokens, "cut_off": cut_off,
            "solution": sol or "", "answer": ans or "", "answered": bool(ans and ans.strip()),
            "none": bool(ans and _NONE.fullmatch(ans)), "class": klass(sol, ans, p["numbers"], p["target"]),
            "solution_tokens": _ntok(sol)}

_csv = f"{PROBE_DIR}/probe.csv"
if not PROMPT_PROBE:
    print("[prompt probe] off (PROMPT_PROBE = False)")
elif os.path.exists(_csv):
    probe = pd.read_csv(_csv, keep_default_na=False)
    print(f"[prompt probe] loaded {_csv}")
else:
    load_policy(S_ADAPTER)
    llm = trainer.vllm_generation.llm
    t0 = time.time()
    rows = []
    for name, suffix in PROBE_PROMPTS.items():
        ids = [chat_ids(_probe_prompt(p, suffix)) for p in _pp]
        outs = llm.generate([{"prompt_token_ids": x} for x in ids],
                            SamplingParams(n=1, temperature=TEMPERATURE, top_p=1.0, max_tokens=MAX_COMPLETION_LENGTH, seed=SEED),
                            use_tqdm=False)
        rows += [_row(p, name, "normal", o.outputs[0].text, len(o.outputs[0].token_ids), o.outputs[0].finish_reason == "length")
                 for p, o in zip(_pp, outs)]
        pre = prefix("")
        ids = [x + tok(pre, add_special_tokens=False)["input_ids"] for x in ids]
        outs = llm.generate([{"prompt_token_ids": x} for x in ids],
                            SamplingParams(n=1, temperature=TEMPERATURE, top_p=1.0, max_tokens=PROBE_ROOM, seed=SEED + 1,
                                           stop=["</answer>"], include_stop_str_in_output=True), use_tqdm=False)
        for p, o in zip(_pp, outs):
            g = o.outputs[0]
            r = _row(p, name, "empty", pre + g.text, len(g.token_ids), not g.text.rstrip().endswith("</answer>"))
            r["cut_off_at_room"] = r["cut_off"] or len(g.token_ids) > ROOM
            rows.append(r)
    probe = pd.DataFrame(rows)
    items = [(json.loads(nu), int(t), judge_text(s, a) if (a.strip() and s.strip() and st <= MAX_SOLUTION_TOKENS) else None)
             for nu, t, s, a, st in zip(probe.numbers, probe.target, probe.solution, probe.answer, probe.solution_tokens)]
    probe["judge_mean"], probe["judge_sum"] = judge_scores(trainer.model, items)
    probe.to_csv(_csv, index=False)
    print(f"[prompt probe] {len(probe)} completions in {(time.time() - t0) / 60:.1f} min; saved {_csv}")

if PROMPT_PROBE:
    for c in ("cut_off", "answered", "none"):
        probe[c] = probe[c].astype(str) == "True"
    probe["cut_off_at_room"] = probe.get("cut_off_at_room", pd.Series(False, index=probe.index)).astype(str) == "True"
    probe["judge_sum"] = pd.to_numeric(probe.judge_sum, errors="coerce")
    probe["judge_mean"] = pd.to_numeric(probe.judge_mean, errors="coerce")
    g = probe.groupby(["prompt", "side", "n"], sort=False)
    tab = pd.DataFrame({"items": g.size(), "cut off": g.cut_off.mean(), f"cut off at room {ROOM}": g.cut_off_at_room.mean(),
                        "answered": g.answered.mean(), "none": g.none.mean(),
                        "correct": g["class"].apply(lambda c: (c == "correct").mean()),
                        "honest miss": g["class"].apply(lambda c: (c == "honest miss").mean()),
                        "fake": g["class"].apply(lambda c: (c == "fake").mean()),
                        "tokens median": g.tokens.median(),
                        "solution tokens p90": g.solution_tokens.quantile(0.9),
                        f"solution > {MAX_SOLUTION_TOKENS}": g.solution_tokens.apply(lambda s: (s > MAX_SOLUTION_TOKENS).mean())})
    tab.loc[tab.index.get_level_values("side") == "normal", f"cut off at room {ROOM}"] = np.nan
    js = (probe[probe.judge_sum.notna()].groupby(["prompt", "side", "class"], sort=False)
          .agg(items=("judge_sum", "size"), judge_sum=("judge_sum", "median"), judge_mean=("judge_mean", "median")))
    with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250, "display.max_columns", None):
        print(f"\n[prompt probe] S on {PROBE_N} validation corruption puzzles per size, 1 sample, T {TEMPERATURE:g}; normal: cap "
              f"{MAX_COMPLETION_LENGTH}; empty: room {PROBE_ROOM} ([corruption]'s room was {ROOM}); classes by the checker\n"
              + tab.to_string())
        print("\n[prompt probe] judge (DPO adapter) on scored items, median by answer class\n" + js.to_string())
    _rng = np.random.default_rng(SEED)
    out = []
    for (name, side), d in probe.groupby(["prompt", "side"], sort=False):
        by = {c: list(_rng.permutation(x.index)) for c, x in d.groupby("class")}
        pick = []
        while len(pick) < 5 and any(by.values()):
            for c in sorted(by, key=lambda c: -len(d[d["class"] == c])):
                if by[c] and len(pick) < 5:
                    pick.append(by[c].pop())
        out.append(f"\n{'#' * 100}\n# prompt {name!r}, {side}\n{'#' * 100}")
        for i in pick:
            r = probe.loc[i]
            body = r.text if side == "empty" else r.text[r.text.find("</reasoning>"):] if "</reasoning>" in r.text else r.text[-1500:]
            out.append(f"\n--- {r.n}-number, target {r.target}, numbers {r.numbers} | {r['class']} | tokens {r.tokens} | "
                       f"cut off {r.cut_off} | judge sum {r.judge_sum:.1f}\n{body.strip()[:2500]}")
    review = "\n".join(out)
    open(f"{PROBE_DIR}/review.txt", "w", encoding="utf-8").write(review)
    print(review)


In [ ]:
# [corruption] S (in the engine) writes one normal completion per corruption puzzle (GRPO prompt, T, cap); for each usable
# one (finished, solution scored by the GRPO reward) and each kind, S writes only <solution> and <answer> after the
# forced "<reasoning>\n<kept>\n</reasoning>\n<solution>\n": empty = no reasoning, cut<f> = the first f of its own
# reasoning's tokens back to the last whole line (or word). Room: SOL_MAX_TOKENS = the normal spans' 99th percentile.
# Saves CORRUPT_DIR/normal.csv and corrupt.csv (skipped when both exist).
REASONING = re.compile(r"\s*<reasoning>\n?(.*?)\n?</reasoning>\s*(?=<solution>)", re.DOTALL)
NORMAL_PATH, CORRUPT_PATH = f"{CORRUPT_DIR}/normal.csv", f"{CORRUPT_DIR}/corrupt.csv"

def prefix(reasoning):
    return "<reasoning>\n" + (f"{reasoning}\n" if reasoning else "") + "</reasoning>\n<solution>\n"

def cut(reasoning, frac):
    ids = tok(reasoning, add_special_tokens=False)["input_ids"]
    text = tok.decode(ids[:int(frac * len(ids))])
    i = text.rfind("\n") if "\n" in text else text.rstrip().rfind(" ")
    return text[:max(i, 0)].rstrip()

if os.path.exists(NORMAL_PATH) and os.path.exists(CORRUPT_PATH):
    print(f"[corruption] skipped: {NORMAL_PATH} and {CORRUPT_PATH} exist")
else:
    load_policy(S_ADAPTER)
    POLICY = "S"
    llm = trainer.vllm_generation.llm
    t0 = time.time()
    _ids = [chat_ids(countdown_prompt(p["numbers"], p["target"])) for p in corrupt_puzzles]
    _outs = llm.generate([{"prompt_token_ids": x} for x in _ids],
                         SamplingParams(n=1, temperature=TEMPERATURE, top_p=1.0, max_tokens=MAX_COMPLETION_LENGTH, seed=SEED))
    assert all(list(o.prompt_token_ids) == x for o, x in zip(_outs, _ids)), "vLLM returned outputs for other prompts"
    rows = []
    for p, o in zip(corrupt_puzzles, _outs):
        t = o.outputs[0].text
        m = REASONING.match(t)
        sol, ans = tag(t, "solution"), tag(t, "answer")
        i = t.find("<solution>\n", m.end()) if m else -1
        j = t.find("</answer>", i) if i >= 0 else -1
        rows.append({"pid": p["pid"], "n": p["n"], "val": p["val"], "numbers": json.dumps(p["numbers"]),
                     "target": p["target"], "completion": t, "tokens": len(o.outputs[0].token_ids),
                     "truncated": o.outputs[0].finish_reason == "length", "reasoning": m.group(1).strip() if m else None,
                     "span_tokens": (len(tok(t[i + len("<solution>\n"):j + len("</answer>")], add_special_tokens=False)["input_ids"])
                                     if m and i >= 0 and j >= 0 else None),
                     "solution": sol, "answer": ans, "class": klass(sol, ans, p["numbers"], p["target"])})
    normal = pd.DataFrame(rows)
    normal["usable"] = [not tr and isinstance(r, str) and pd.notna(st) and scorable(s, a)
                        for tr, r, st, s, a in zip(normal.truncated, normal.reasoning, normal.span_tokens,
                                                   normal.solution, normal.answer)]
    assert normal.usable.any(), "no usable normal completion"
    SOL_MAX_TOKENS = int(math.ceil(normal.loc[normal.usable, "span_tokens"].quantile(0.99) / 8) * 8)
    normal["usable"] &= normal.span_tokens.fillna(10**9) <= SOL_MAX_TOKENS
    print(f"[corruption] normal: {len(normal)} in {(time.time() - t0) / 60:.1f} min, usable {normal.usable.sum()} "
          f"({normal.groupby('n').usable.sum().to_dict()}), truncated {normal.truncated.mean():.2f}; room {SOL_MAX_TOKENS} tokens")
    jobs = []
    for _, r in normal[normal.usable].iterrows():
        for kind in KINDS:
            reasoning = "" if kind == "empty" else cut(r.reasoning, float(kind[3:]))
            jobs.append({"pid": r.pid, "kind": kind, "reasoning_tokens": len(tok(reasoning, add_special_tokens=False)["input_ids"]),
                         "of_tokens": len(tok(r.reasoning, add_special_tokens=False)["input_ids"]),
                         "prefix": prefix(reasoning), "numbers": r.numbers, "target": r.target, "n": r.n, "val": r.val})
    _ids = [chat_ids(countdown_prompt(json.loads(j["numbers"]), int(j["target"]))) + tok(j["prefix"], add_special_tokens=False)["input_ids"]
            for j in jobs]
    _outs = llm.generate([{"prompt_token_ids": x} for x in _ids],
                         SamplingParams(n=1, temperature=TEMPERATURE, top_p=1.0, max_tokens=SOL_MAX_TOKENS, seed=SEED + 1,
                                        stop=["</answer>"], include_stop_str_in_output=True))
    assert all(list(o.prompt_token_ids) == x for o, x in zip(_outs, _ids)), "vLLM returned outputs for other prompts"
    for j, o in zip(jobs, _outs):
        full = j["prefix"] + o.outputs[0].text
        sol, ans = tag(full, "solution") or "", tag(full, "answer") or ""
        j.update({"generated": o.outputs[0].text, "finished": o.outputs[0].text.rstrip().endswith("</answer>"),
                  "solution": sol, "answer": ans, "class": klass(sol, ans, json.loads(j["numbers"]), j["target"]),
                  "budget": SOL_MAX_TOKENS})
        j["usable"] = j["finished"] and scorable(sol, ans)
    corrupt = pd.DataFrame(jobs).drop(columns=["prefix"])
    normal.to_csv(NORMAL_PATH, index=False)
    corrupt.to_csv(CORRUPT_PATH, index=False)
    print(f"[corruption] saved {NORMAL_PATH} and {CORRUPT_PATH} ({len(corrupt)} corrupted) in {(time.time() - t0) / 60:.1f} min")
normal = _rd(NORMAL_PATH)
corrupt = _rd(CORRUPT_PATH)
for d, cols in ((normal, ("usable", "truncated", "val")), (corrupt, ("finished", "usable", "val"))):
    for c in cols:
        d[c] = _b(d, c)
_u = normal[normal.usable].assign(kind="normal")
_tab = pd.concat([_u[["kind", "n", "class"]], corrupt[corrupt.usable][["kind", "n", "class"]]])
with pd.option_context("display.width", 200, "display.float_format", "{:.3f}".format):
    print("answer classes of usable completions (checker, measurement only):\n"
          + pd.crosstab([_tab.kind, _tab.n], _tab["class"], normalize="index").to_string())
    print("usable:", _tab.groupby(["kind", "n"]).size().to_dict(), "| corrupted finished:",
          corrupt.groupby("kind").finished.mean().round(3).to_dict())


Rendering prompts:   0%|          | 0/2000 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 2000/2000 [04:23<00:00,  7.58it/s, est. speed input: 989.51 toks/s, output: 8084.77 toks/s]


[corruption] normal: 2000 in 4.4 min, usable 1276 ({3: 788, 4: 488}), truncated 0.34; room 120 tokens


Rendering prompts:   0%|          | 0/5104 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 5104/5104 [00:53<00:00, 94.82it/s, est. speed input: 21736.83 toks/s, output: 7811.64 toks/s] 


[corruption] saved /content/drive/MyDrive/Consistency/countdown/sft/Qwen3-0.6B-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500/judge_corrupt_empty-cut0.1-cut0.25-cut0.5_unfinished_n4x1000_n3x1000_t0.9_cap2048_seed0/normal.csv and /content/drive/MyDrive/Consistency/countdown/sft/Qwen3-0.6B-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500/judge_corrupt_empty-cut0.1-cut0.25-cut0.5_unfinished_n4x1000_n3x1000_t0.9_cap2048_seed0/corrupt.csv (5104 corrupted) in 5.6 min
answer classes of usable completions (checker, measurement only):
class      bad answer  correct  fake  honest miss  other  wrong numbers
kind    n                                                              
cut0.1  3       0.005    0.110 0.435        0.148  0.282          0.019
        4       0.000    0.020 0.647        0.108  0.225          0.000
cut0.25 3       0.000    0.081 0.423        0.235  0.217          0.043
        4       0.012    0.024 0.661        0.117  0.144      

In [ ]:
# [pairs] Consistency_Corrupt_Judge_Olmo's: chosen = the normal solution + answer, rejected = the solution after corrupted
# reasoning (same puzzle): answered ones as solution + answer, with KEEP_UNFINISHED unfinished ones as written inside
# <solution>. Identical texts dropped. Training: TRAIN_KINDS on the training puzzles; validation: every kind on the last
# VAL_SHARE. The checker only measures (label right / inverted / both correct / both wrong).
_nu = normal[normal.usable].set_index("pid")
rej = corrupt[corrupt.kind.isin(KINDS)].copy()
rej["answered"] = rej.usable
rej["unfinished"] = ~rej.finished
rej["rejected_text"] = [judge_text(s, a) if u else unfinished_text(g)
                        for s, a, g, u in zip(rej.solution, rej.answer, rej.generated, rej.answered)]
keep = rej.answered | (KEEP_UNFINISHED & rej.unfinished & (rej.rejected_text != ""))
pairs_all = rej[keep].join(_nu[["solution", "answer", "class"]], on="pid", rsuffix="_normal")
pairs_all = pairs_all[pairs_all.solution_normal.notna()]
pairs_all["chosen_text"] = [judge_text(s, a) for s, a in zip(pairs_all.solution_normal, pairs_all.answer_normal)]
pairs_all["identical"] = pairs_all.chosen_text == pairs_all.rejected_text
pairs_all["correct_normal"] = pairs_all.class_normal == "correct"
pairs_all["correct_rejected"] = pairs_all["class"] == "correct"
pairs_all["label"] = [{(True, False): "right", (False, True): "inverted", (True, True): "both correct",
                       (False, False): "both wrong"}[(a, b)] for a, b in zip(pairs_all.correct_normal, pairs_all.correct_rejected)]
_t = pairs_all.groupby(["n", "kind"]).agg(pairs=("pid", "size"), unfinished=("unfinished", "mean"),
                                          rejected_correct=("correct_rejected", "mean"), identical=("identical", "mean"))
_t = _t.join(pd.crosstab([pairs_all.n, pairs_all.kind], pairs_all.label, normalize="index"))
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print("pairs per size and kind:\n" + _t.to_string())
_t.to_csv(f"{CORRUPT_DIR}/pairs_check.csv")

def to_dpo(d):
    return [{"prompt": judge_messages(json.loads(r.numbers), int(r.target)),
             "chosen": [{"role": "assistant", "content": r.chosen_text}],
             "rejected": [{"role": "assistant", "content": r.rejected_text}],
             **({"chat_template_kwargs": CHAT_KWARGS} if CHAT_KWARGS else {}),
             "kind": r.kind, "n": int(r.n), "unfinished": bool(r.unfinished), "label": r.label,
             "numbers": r.numbers, "target": int(r.target)} for r in d.itertuples()]

_p = pairs_all[~pairs_all.identical]
train_pairs = to_dpo(_p[~_p.val & _p.kind.isin(TRAIN_KINDS)].sample(frac=1, random_state=SEED))
val_pairs = to_dpo(_p[_p.val])
print(f"\n{len(train_pairs)} training pairs, {len(val_pairs)} validation pairs; e.g.\nCHOSEN\n"
      f"{train_pairs[0]['chosen'][0]['content']}\nREJECTED\n{train_pairs[0]['rejected'][0]['content']}")


pairs per size and kind:
           pairs  unfinished  rejected_correct  identical  both correct  both wrong  inverted  right
n kind                                                                                              
3 cut0.1     784       0.467             0.059      0.000         0.032       0.538     0.027  0.403
  cut0.25    788       0.358             0.052      0.003         0.023       0.533     0.029  0.415
  cut0.5     788       0.321             0.029      0.016         0.028       0.561     0.001  0.410
  empty      598       0.863             0.015      0.000         0.008       0.580     0.007  0.405
4 cut0.1     488       0.490             0.010      0.000         0.002       0.816     0.008  0.174
  cut0.25    488       0.318             0.016      0.000         0.000       0.807     0.016  0.176
  cut0.5     488       0.346             0.020      0.016         0.010       0.814     0.010  0.166
  empty      305       0.898             0.007      0.000         

In [ ]:
# [judge DPO] Continue S's adapter with DPO on the training pairs (reference = S, TRL's frozen "ref" copy), batch 16,
# cosine, 5% warmup, on a separate copy of the model (the trainer's stays as it is); saves JUDGE_PATH (skipped when it
# exists). Then the judge is loaded into the trainer's model as a third, frozen adapter "judge".
from peft import PeftModel
if os.path.exists(f"{JUDGE_PATH}/adapter_model.safetensors"):
    print(f"[judge DPO] skipped: {JUDGE_PATH} exists")
else:
    from trl import DPOConfig, DPOTrainer
    _cols = ("prompt", "chosen", "rejected") + (("chat_template_kwargs",) if CHAT_KWARGS else ())
    _base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="auto")
    _base.config.pad_token_id = tok.pad_token_id
    dpo_model = PeftModel.from_pretrained(_base, S_ADAPTER, is_trainable=True)
    _steps = -(-len(train_pairs) // 16) * DPO_EPOCHS
    dpo_args = DPOConfig(
        output_dir=f"{JUDGE_DIR}/checkpoints", beta=DPO_BETA, learning_rate=DPO_LR, num_train_epochs=DPO_EPOCHS,
        lr_scheduler_type="cosine", warmup_steps=max(1, round(0.05 * _steps)),
        per_device_train_batch_size=4, gradient_accumulation_steps=4, per_device_eval_batch_size=4,
        max_length=DPO_MAX_LENGTH, bf16=True, gradient_checkpointing=True, logging_steps=10, save_strategy="no",
        eval_strategy="steps", eval_steps=20, report_to="none", seed=SEED, **({"max_steps": 2} if SMOKE else {}))
    dpo_trainer = DPOTrainer(model=dpo_model, args=dpo_args, processing_class=tok,
                             train_dataset=Dataset.from_list([{k: x[k] for k in _cols} for x in train_pairs]),
                             eval_dataset=Dataset.from_list([{k: x[k] for k in _cols} for x in val_pairs]))
    assert set(dpo_model.peft_config) == {"default", "ref"}, list(dpo_model.peft_config)
    _r0 = dpo_trainer.train_dataset[0]       # the judge's view: judge prompt (thinking off) + text + end of turn
    _full = tok.decode(_r0["prompt_ids"] + _r0["chosen_ids"]) if "prompt_ids" in _r0 else None
    if _full is not None:
        assert _full.startswith(tok.decode(chat_ids(train_pairs[0]["prompt"]))), _full[:400]
        print("[judge DPO] tokenized example ends:", repr(_full[-200:]))
    print(f"{len(train_pairs)} pairs, {_steps} optimizer steps; reference = S")
    dpo_trainer.train()
    dpo_model.save_pretrained(JUDGE_PATH, selected_adapters=["default"])
    if os.path.isdir(f"{JUDGE_PATH}/default"):
        for f in os.listdir(f"{JUDGE_PATH}/default"):
            shutil.move(f"{JUDGE_PATH}/default/{f}", f"{JUDGE_PATH}/{f}")
        os.rmdir(f"{JUDGE_PATH}/default")
    assert os.path.exists(f"{JUDGE_PATH}/adapter_model.safetensors"), sorted(os.listdir(JUDGE_PATH))
    _log = pd.DataFrame(dpo_trainer.state.log_history)
    _log.to_csv(f"{JUDGE_DIR}/train_log.csv", index=False)
    _show = [c for c in ("step", "loss", "eval_loss", "eval_rewards/accuracies", "eval_rewards/margins") if c in _log]
    print(_log[_show].dropna(how="all", subset=_show[1:]).to_string(index=False))
    del dpo_trainer, dpo_model, _base
    gc.collect(); torch.cuda.empty_cache()
if "judge" not in trainer.model.peft_config:
    trainer.model.load_adapter(JUDGE_PATH, adapter_name="judge", is_trainable=False)
    trainer.model.set_adapter("default")
    _freeze_others(trainer.model)
assert all(p.requires_grad == (".default." in n) for n, p in trainer.model.named_parameters() if "lora_" in n)
print(f"[judge DPO] adapters in the trainer's model: {sorted(trainer.model.peft_config)} (only default trains)")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/808 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/808 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/486 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/486 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


[judge DPO] tokenized example ends: 'use +, -, * and /, and each number must be used exactly once.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n31 + 25 = 56\n56 - 41 = 15\n15 * 3 = 45\n45 + 2 = 47\nAnswer: (31 + 25) - 41 + 3<|im_end|>\n'
808 pairs, 102 optimizer steps; reference = S


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: FutureWarning: `torch.distributed.all_gather_into_tensor` is deprecated. Please use `torch.distributed.all_gather_single` instead.
  return func(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
20,0.074209,0.239815,0.270436,104996.000000,-2.023676,-2.511137,0.846820,2.533831,-0.386118,0.895492,2.919949,-24.986360,-64.165308
40,0.027565,0.269877,0.251771,209961.000000,-2.772116,-3.178803,0.838710,1.887474,-2.106184,0.879098,3.993658,-31.449935,-81.365975
60,0.018684,0.230826,0.247736,312223.000000,-2.916950,-3.296348,0.851842,2.375429,-1.730766,0.901639,4.106195,-26.570385,-77.611791
80,0.024034,0.230054,0.241404,416118.000000,-3.081275,-3.461465,0.847972,2.150010,-2.205177,0.895492,4.355188,-28.824570,-82.355906
100,0.020746,0.231739,0.240503,521391.000000,-3.115882,-3.493964,0.847518,2.117685,-2.263844,0.891393,4.381529,-29.147825,-82.942571
102,0.020746,0.230070,0.240503,529334.000000,-3.119515,-3.494441,0.847329,2.117107,-2.263520,0.891393,4.380628,-29.153600,-82.939336


 step     loss  eval_loss  eval_rewards/accuracies  eval_rewards/margins
   10 0.346102        NaN                      NaN                   NaN
   20 0.074209        NaN                      NaN                   NaN
   20      NaN   0.239815                 0.895492              2.919949
   30 0.039950        NaN                      NaN                   NaN
   40 0.027565        NaN                      NaN                   NaN
   40      NaN   0.269877                 0.879098              3.993658
   50 0.048674        NaN                      NaN                   NaN
   60 0.018684        NaN                      NaN                   NaN
   60      NaN   0.230826                 0.901639              4.106195
   70 0.045628        NaN                      NaN                   NaN
   80 0.024034        NaN                      NaN                   NaN
   80      NaN   0.230054                 0.895492              4.355188
   90 0.021890        NaN                      NaN 

In [ ]:
# [calibrate] S's two-pass rollouts on the grade puzzles (CALIB_SAMPLES per puzzle and size, the training rollout:
# pass 1 to the cap, forced solution when truncated before <solution>), each scored as the GRPO reward would (judge_input)
# by the judge, S itself ("ref") and the untrained model (adapters off). Then
#   the self-judge check: within-puzzle win rate of correct over wrong final answers (checker, measurement only) per size
#     and source (natural finished / forced / all), for the judge (sum, mean), S (sum, mean), the untrained model, the
#     log-ratio judge - S (sum) and "shorter" (fewer judged tokens);
#   the judge's weight = within-puzzle SD of the format reward / within-puzzle SD of the judge score (each: the square
#     root of the mean within-puzzle variance, scored rollouts), missing = the judge score's 5th percentile.
# Saved to CALIB_PATH (+ the rollouts' scores next to it); skipped when it exists.
_cal_csv = CALIB_PATH.replace(".json", "_rollouts.csv")
if os.path.exists(CALIB_PATH):
    cal = json.load(open(CALIB_PATH))
    ev = _rd(_cal_csv)
    for c in ("correct", "forced", "scored", "unfinished"):
        ev[c] = _b(ev, c)
    print(f"[calibrate] loaded {CALIB_PATH}")
else:
    load_policy(S_ADAPTER)
    POLICY = "S"
    t0 = time.time()
    meta = [(n, p) for n in (4, 3) for p in GRADE[n] for _ in range(CALIB_SAMPLES)]
    ro = two_pass_rollout([countdown_prompt(p["numbers"], p["target"]) for _, p in meta], trainer)
    trainer._metrics["train"].clear()
    final = [tok.decode(c, skip_special_tokens=True) for c in ro["completion_ids"]]
    items = [(p["numbers"], p["target"], judge_input(x, f, g)) for (_, p), x, f, g in
             zip(meta, ro["natural"], ro["forced"], ro["forced_gen"])]
    ev = pd.DataFrame({"n_numbers": [n for n, _ in meta], "puzzle": [p["puzzle"] for _, p in meta],
                       "numbers": [json.dumps(p["numbers"]) for _, p in meta], "target": [p["target"] for _, p in meta],
                       "forced": ro["forced"], "text": [x or "" for _, _, x in items],
                       "format_reward": format_total(ro["natural"]),
                       "correct": np.array(R.correctness_reward_func(None, final, [p["numbers"] for _, p in meta],
                                                                     [p["target"] for _, p in meta])) > 0})
    ev["scored"] = ev.text != ""
    ev["unfinished"] = ev.scored & ~ev.text.str.contains("\nAnswer: ", regex=False)
    for name, adapter in (("judge", "judge"), ("S", "ref"), ("base", None)):
        m, s = judge_scores(trainer.model, items, adapter=adapter)
        ev[f"{name}_mean"], ev[f"{name}_sum"] = m, s
    ev["ratio"] = ev.judge_sum - ev.S_sum
    ev["shorter"] = [-len(tok(x, add_special_tokens=False)["input_ids"]) if x else np.nan for x in ev.text]
    sc = ev[ev.scored]
    sd_within = lambda d, col: float(np.sqrt(d.groupby(["n_numbers", "puzzle"])[col].var(ddof=0).mean()))
    key = f"judge_{JUDGE_REDUCTION}"
    cal = {"format_sd_within": sd_within(sc, "format_reward"), "judge_sd_within": sd_within(sc, key),
           "missing": float(np.percentile(sc[key], 5)), "reduction": JUDGE_REDUCTION, "rollouts": len(ev),
           "scored": int(ev.scored.sum()), "judge": JUDGE_PATH}
    cal["weight"] = cal["format_sd_within"] / cal["judge_sd_within"]
    ev.to_csv(_cal_csv, index=False)
    json.dump(cal, open(CALIB_PATH, "w"), indent=1)
    print(f"[calibrate] {len(ev)} rollouts in {(time.time() - t0) / 60:.1f} min; saved {CALIB_PATH}")
JUDGE_WEIGHT, JUDGE_MISSING = cal["weight"], cal["missing"]
print(f"[calibrate] judge {cal['reduction']}: weight {JUDGE_WEIGHT:.4g} (format SD {cal['format_sd_within']:.3f} / judge SD "
      f"{cal['judge_sd_within']:.3f}), missing {JUDGE_MISSING:.1f}; scored {cal['scored']} of {cal['rollouts']}")

METRICS = ["judge_sum", "judge_mean", "ratio", "S_sum", "S_mean", "base_sum", "base_mean", "shorter"]
rows = []
for size in (4, 3):
    d = ev[(ev.n_numbers == size) & ev.scored]
    for lab, x in {"natural (finished)": d[~d.forced & ~d.unfinished], "forced": d[d.forced], "all scored": d}.items():
        r = {"numbers": size, "set": lab, "items": len(x), "correct": x.correct.mean()}
        for m in METRICS:
            r[m], r["pairs"] = within_puzzle_win_rate((x.n_numbers.astype(str) + "-" + x.puzzle.astype(str)).tolist(),
                                                      x[m].values.astype(float), x.correct.values.astype(bool))
        r["puzzles with both"] = int((x.groupby("puzzle").correct.agg(lambda c: c.any() and not c.all())).sum())
        rows.append(r)
check = pd.DataFrame(rows)
check.to_csv(CALIB_PATH.replace(".json", "_winrates.csv"), index=False)
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250, "display.max_columns", None):
    print(f"\n[self-judge check] S's two-pass rollouts on the grade puzzles ({CALIB_SAMPLES} per puzzle, T {TEMPERATURE:g}, "
          f"cap {MAX_COMPLETION_LENGTH}, forced room {FORCE_ROOM}): within-puzzle win rate of correct over wrong (ties 1/2; "
          f"0.5 = no signal). judge = DPO judge, S = the SFT start itself, base = untrained {MODEL_KEY}\n"
          + check.to_string(index=False))


In [ ]:
# [release] All results are on Drive: release the runtime if UNASSIGN_WHEN_DONE (a failed cell stops Run all before this).
if UNASSIGN_WHEN_DONE:
    from google.colab import runtime
    runtime.unassign()
